#Clustering em Transcriptômica Espacial: Visium HD em Espermatogênese

Este notebook detalha a aplicação do pipeline validado no estudo de caso da Síndrome de Células de Sertoli Puras (SCOS). Através da utilização de objetos SpatialData/Zarr, as amostras PC (Controle) e PM (Mutada) são analisadas de forma comparativa. O foco reside na identificação de nichos celulares específicos, na caracterização da falha germinativa e na quantificação da hiperplasia de linhagens de suporte no microambiente testicular.
A resolução de 2 µm da plataforma Visium HD foi determinante para distinguir a transição sutil entre as células somáticas e os estágios iniciais de maturação germinativa, algo inviável em resoluções espaciais convencionais (55 µm)

# **1. Bibliotecas**


In [ ]:
%%capture

!pip install uv
!uv pip install --system anndata==0.12.0 pydeseq2==0.5.2 squidpy==1.6.5 "spatialdata[extra]==0.4.0" geosketch==1.3 harmonypy==0.0.10 igraph==0.11.8 spatialdata-plot datashader dask-image "numpy<2" leidenalg igraph gseapy -q

In [ ]:
%%time

import numpy as np
import pandas as pd
import json
import gc #gerenciamento de memória RAM

import spatialdata as spd
import spatialdata_plot as splt
import spatialdata_io as so
import scanpy as sc
import scanpy.external as sce

import geopandas as gpd
from spatialdata.models import Image2DModel, TableModel, ShapesModel
from shapely.geometry import Polygon
from PIL import Image

import geosketch as sketch #downsampling de dados massivos
from pydeseq2.dds import DeseqDataSet #análise de expressão diferencial
from pydeseq2.ds import DeseqStats

import matplotlib.pyplot as plt
import seaborn as sns #necessário para o heatmap
from spatialdata.transformations import Identity, Scale

from google.colab import drive
import os
from IPython.display import Image as ShowImage, display
import requests
from scipy.stats import hypergeom
import urllib.request
import zipfile
import gseapy as gp

In [ ]:
#monta drive
drive.mount('/content/drive')

# **2. Ingestão de dados**

O dataset utilizado neste trabalho foi disponibilizado em nuvem pela pesquisadora e compreende dados de dois pacientes:
* Paciente com mutação [PM]: amostra de tecido testicular de paciente com mutação no gene MYRF e Síndrome de Células de Sertoli Puras [SCOS], descrito por [Calonga-Solís et al. (2022)](https://www.mdpi.com/2077-0383/11/16/4858);

* Paciente controle [PC]: amostra de tecido testicular de paciente sem alterações.

Os dados foram processados pelo pipeline spaceranger count v4.0.1. via [Cloud Analysis](https://).

Para evitar estourar a memória do disco local do Colab, os arquivos foram salvos  no Google Drive.

## 2.1. Paciente [PM]

In [ ]:
#monta drive
drive.mount('/content/drive')

#define o caminho no Google Drive para salvar os dados do paciente
caminho_projeto = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/patient'

#muda para a pasta de dados do paciente
%cd {caminho_projeto}

#verificação
!pwd

In [ ]:
%%capture

#importação dos outputs do 10x Genomics Cloud Analysis para o Google Drive
!wget -O metrics_summary.csv "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/92c9653c-6b6d-4f9c-a4dc-120a7262540b/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O probe_set.csv "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/f950cf8c-8f88-4ef2-a558-45ad2add0018/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O web_summary.html "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/4e5691fb-d900-4c4b-b1ad-69342938564d/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O spatial.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/eb016afe-3414-4955-813c-1e795880e025/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O barcode_mappings.parquet "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/d9831217-7d3b-49f0-a0e6-2a267372e690/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O molecule_info.h5 "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/cd9892dc-a6bd-4d19-a683-94e2b12358ce/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O feature_slice.h5 "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/b66c0455-d481-40c6-8290-8a8beb77defa/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O segmented_outputs.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/34ba5dff-e071-454d-9bdf-37f4facda1b2/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O binned_outputs.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/7d02eb0b-5af1-455e-96f4-59be483321f0/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O cloupe_008um.cloupe "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/10f14758-175d-4837-94e7-7e9473e3c152/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"
!wget -O cloupe_cell.cloupe "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/53ae9389-17d4-4d6d-8982-3673269d44aa/?token=RsgP0cffug3th_h7dEOeyY248hajUxoW"

In [ ]:
#extrai o conteúdo do spatial.tar.gz
!tar -xzf spatial.tar.gz

In [ ]:
#visualização da amostra PM
caminho_imagem = 'tissue_hires_image.png'
display(ShowImage(filename=caminho_imagem, width=600))

In [ ]:
#xtrair segmentações para o paciente
%cd /content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/patient
!tar -xzf segmented_outputs.tar.gz

##**2.2. Controle [PC]**

In [ ]:
#define o caminho no Google Drive para salvar os dados do controle
caminho_projeto_ctrl = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/control'

#muda para a pasta de dados do controle
%cd {caminho_projeto_ctrl}

#verificação
!pwd

In [ ]:
%%capture

#importação dos resultados do 10x Genomics Cloud Analysis para o Google Drive
!wget -O metrics_summary.csv "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/3da55553-830e-4ec8-a6f3-c6ad35581d66/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O probe_set.csv "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/2bba4cd3-28ec-4e9f-a435-db1caad94fc9/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O web_summary.html "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/3fd52ac9-1dc6-44e2-9233-b9cf11955370/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O spatial.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/f8363ca2-f900-42e6-9602-21f70e99f600/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O barcode_mappings.parquet "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/2e519977-afaa-4f87-9eb9-69b0675befdb/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O molecule_info.h5 "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/5624245a-0266-4774-931a-a24f9c715ae3/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O feature_slice.h5 "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/54baf74f-c7b5-4d88-8ec9-a34f3d5aa8ea/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O segmented_outputs.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/8fd6d6e5-8927-43c0-9413-42096c64c0c7/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O binned_outputs.tar.gz "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/165e5795-fa90-49ef-baee-fec5e19ff28c/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O cloupe_008um.cloupe "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/43b97c08-b8f5-4099-bee1-3ce162bd3a43/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"
!wget -O cloupe_cell.cloupe "https://cloud.10xgenomics.com/api/cloud-analysis/downloads-api/v1/780638c9-4cdc-4097-b664-d59a881cc96e/?token=f-Me9QgLgmmkXincrsyAG3L5qYYCUeIv"

In [ ]:
#extrai o conteúdo do spatial.tar.gz
!tar -xzf spatial.tar.gz

In [ ]:
#visualização da amostra PC

caminho_imagem_ctrl = 'tissue_hires_image.png'
display(ShowImage(filename=caminho_imagem_ctrl, width=600))

In [ ]:
#extrair segmentações para o controle
%cd /content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/control
!tar -xzf segmented_outputs.tar.gz

#**3. Data Wrangling**

##**3.1. Funções auxiliares**

###**3.1.1. create_zarr**

In [ ]:
def create_zarr(count_matrix_path,   #caminho para o arquivo da matriz de contagem
                image_path,          #caminho para o arquivo de imagem
                scale_factors_path,  #caminho para o arquivo JSON dos fatores de escala
                geojson_path,        #caminho para o arquivo GeoJSON de segmentação das células
                sample_name          #nome para o arquivo de saída do Zarr
):
    # 1. recebe caminhos de entrada
    print(sample_name)

    # 2. carrega e prepara os dados
    COUNT_MATRIX_PATH = count_matrix_path
    IMAGE_PATH = image_path
    SCALE_FACTORS_PATH = scale_factors_path
    GEOJSON_PATH = geojson_path

    # Load AnnData
    adata = sc.read_10x_h5(COUNT_MATRIX_PATH)
    adata.var_names_make_unique()
    adata.obs['sample'] = sample_name

    # AJUSTE para o mapeamento do GeoJSON
    adata.obs.index = sample_name + "_" + adata.obs.index.astype(str)

    # Load and preprocess image data
    image_data = np.array(Image.open(IMAGE_PATH))
    if image_data.ndim == 2:
        image_data = image_data[np.newaxis, :, :] # Add channel dimension for grayscale
    elif image_data.ndim == 3:
        image_data = np.transpose(image_data, (2, 0, 1)) # (H, W, C) -> (C, H, W) for spatialdata

    # Load scale factors
    with open(SCALE_FACTORS_PATH, 'r') as f:
        scale_data = json.load(f)

    # Load GeoJSON data
    with open(GEOJSON_PATH, 'r') as f:
        geojson_data = json.load(f)

    # 3. define sistemas de coordenadas e transformações
    hires_scale = scale_data['tissue_hires_scalef']
    shapes_transformations = {
       "downscale_to_hires": Scale(np.array([hires_scale, hires_scale]), axes=("x", "y"))
    }
    image_transformations = {
        "downscale_to_hires": Identity()
    }

    # 4. processa a segmentação celular
    geojson_features_map = {
        f"{sample_name}_cellid_{feature['properties']['cell_id']:09d}-1": feature
        for feature in geojson_data['features']
    }

    geometries = []
    cell_ids_ordered = []

    for obs_index_str in adata.obs.index:
        feature = geojson_features_map.get(obs_index_str)
        if feature:
            polygon_coords = np.array(feature['geometry']['coordinates'][0])
            geometries.append(Polygon(polygon_coords))
            cell_ids_ordered.append(obs_index_str)
        else:
            geometries.append(None)
            cell_ids_ordered.append(obs_index_str)

    valid_indices = [i for i, geom in enumerate(geometries) if geom is not None]
    geometries = [geometries[i] for i in valid_indices]
    cell_ids_ordered = [cell_ids_ordered[i] for i in valid_indices]

    # 5. integra a segmentação celular ao objeto AnnData
    shapes_gdf = gpd.GeoDataFrame({
        'cell_id': cell_ids_ordered,
        'geometry': geometries
    }, index=cell_ids_ordered)

    # AJUSTE para a compatibilidade de dtypes entre Pandas e SpatialData
    adata.obs['cell_id'] = adata.obs.index.astype(str)

    adata.obs['region'] = sample_name + '_cell_boundaries'
    adata.obs['region'] = adata.obs['region'].astype('category')
    adata = adata[shapes_gdf.index].copy()

    IMAGE_KEY =  sample_name + '_hires_tissue_image'
    TABLE_KEY =  'segmentation_counts'
    SHAPES_KEY = sample_name + '_cell_boundaries'

    # 6. cria objeto SpatialData
    sdata = spd.SpatialData(
        images={
            IMAGE_KEY: Image2DModel.parse(image_data, transformations=image_transformations)
        },
        tables={
            TABLE_KEY: TableModel.parse(
                adata,
                region=SHAPES_KEY,
                region_key='region',
                instance_key='cell_id'
            )
        },
        shapes={
            SHAPES_KEY: ShapesModel.parse(shapes_gdf, transformations=shapes_transformations)
        }
    )

    # 7. grava tudo em um arquivo Zarr.
    sdata.write(sample_name, overwrite=True)
    del sdata
    gc.collect()

###3.1.2. crop0

In [ ]:
def crop0(x,crs,bbox):
    return spd.bounding_box_query(
        x,
        min_coordinate=[bbox['x'][0], bbox['y'][0]],
        max_coordinate=[bbox['x'][1], bbox['y'][1]],
        axes=("x", "y"),
        target_coordinate_system=crs,
    )

##3.2. Processamento em lote (aplicação create_zarr)

In [ ]:
#extrai matrizes binned para o paciente
%cd /content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/patient
!tar -xzf binned_outputs.tar.gz

In [ ]:
#extrai matrizes binned para o controle
%cd /content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/control
!tar -xzf binned_outputs.tar.gz

In [ ]:
%%time

path_PC = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/control'
path_PM = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/data/processed/patient'

# Ajustado com os nomes exatos listados pelo os.listdir
samples = {
    "PC": [f"{path_PC}/filtered_feature_cell_matrix.h5",
           f"{path_PC}/tissue_hires_image.png",
           f"{path_PC}/spatial/scalefactors_json.json",
           f"{path_PC}/cell_segmentations.geojson",
           "PC"],
    "PM": [f"{path_PM}/filtered_feature_cell_matrix.h5",
           f"{path_PM}/tissue_hires_image.png",
           f"{path_PM}/spatial/scalefactors_json.json",
           f"{path_PM}/cell_segmentations.geojson",
           "PM"],
}

# Loop percorre o dicionário e chama create_zarr
print("Saving zarr files")
for key, inputs in samples.items():
    print(f"Processando amostra: {key}")
    create_zarr(count_matrix_path=inputs[0],
                image_path=inputs[1],
                scale_factors_path=inputs[2],
                geojson_path=inputs[3],
                sample_name=inputs[4])

del samples, inputs, key
gc.collect() # Limpeza da RAM

##**3.3. Criação objeto SpatialData**

In [ ]:
%%time

# caminhos dos arquivos Zarr criados anteriormente
visium_hd_zarr_paths = {
    "PC": "./PC",
    "PM": "./PM"
}

# carrega amostras
sdatas = []
for key, path in visium_hd_zarr_paths.items():
    sdata = spd.read_zarr(path)

    for table in sdata.tables.values():
        table.var_names_make_unique()   # garante nomes de genes únicos
        table.obs["sample"] = key       # identificação da amostra

    sdatas.append(sdata)
    del sdata, table
    gc.collect()

# concatena amostras
concatenated_sdata = spd.concatenate(sdatas, concatenate_tables=True)

# carrega as imagens na RAM e revalida o esquema para evitar o SchemaError
for img_name in list(concatenated_sdata.images.keys()):
    # .compute() traz o dado para a RAM; Image2DModel.parse() re-empacota no formato correto
    img_data = concatenated_sdata.images[img_name].compute()
    concatenated_sdata.images[img_name] = Image2DModel.parse(img_data)

# checkpoint sobrescrevendo o anterior no Google Drive
path_spm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm'
os.makedirs(path_spm, exist_ok=True)
concatenated_sdata.write(os.path.join(path_spm, 'sdata_spm_raw.zarr'), overwrite=True)

# limpa variáveis temporárias mas mantém o objeto principal na memória
del sdatas, visium_hd_zarr_paths, key, path
gc.collect()

print("---------------------------------")
print(concatenated_sdata)

## **3.4. Controle da qualidade (QC)**

In [ ]:
# tabela de contagem (AnnData) dentro do objeto consolidado (Spatialdata)
# vinculamos a tabela AnnData à variável adata para facilitar a leitura do código
adata = concatenated_sdata.tables['segmentation_counts']
display(adata)

In [ ]:
# identificação dos genes mitocondriais, que começam com "MT-"
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, percent_top=None)

# visualização do total de UMIs por amostra
plt.figure(figsize=(10, 5)) # proporção da imagem
ax = sc.pl.violin(adata, keys=["log1p_total_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") # rótulo do eixo x
ax.set_ylabel("Contagem de UMI (Log1p)") # rótulo do eixo y
plt.axhline(y=3, color='r', linestyle='--', label="limite inferior") # linhas representam limites sugeridos de corte
plt.axhline(y=8, color='r', linestyle='--', label="limite superior")
plt.title("Distribuição de UMI (Log1p) por Amostra")
plt.show()

# visualização da quantidade de genes por amostra
plt.figure(figsize=(10, 5))
ax = sc.pl.violin(adata, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra")
ax.set_ylabel("Número de genes únicos (Log1p)")
plt.title("Distribuição de genes únicos por amostra")
plt.show()

# visualização da distribuição de genes mitocondriais em números absolutos
plt.figure(figsize=(10, 5))
# Alteração: Correção da chave para "log1p_total_counts_mt" (o original duplicava n_genes) e remoção da linha redundante
ax = sc.pl.violin(adata, keys=["log1p_total_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra")
ax.set_ylabel("Número de genes mitocondriais (Log1p)")
plt.title("Número de genes mitocondriais por amostra")
plt.show()

# visualização do percentual de genes mitocondriais em relação ao total da célula, indicador de morte celular
plt.figure(figsize=(10, 5))
ax = sc.pl.violin(adata, keys=["pct_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra")
ax.set_ylabel("Genes Mitocondriais (%)")
plt.title("Percentual de genes mitocondriais por amostra")
plt.show()

plt.close('all')

In [ ]:
# extração numérica das estatísticas de QC para definir os cortes
qc_stats = adata.obs.groupby('sample')[['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mt']].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).T

# visualização formatada
import pandas as pd
pd.set_option('display.max_rows', None)
print("Estatísticas Descritivas por Amostra:")
display(qc_stats)

#ver os valores de corte baseados em percentis específicos
for smp in adata.obs['sample'].unique():
    subset = adata.obs[adata.obs['sample'] == smp]
    p5 = subset['log1p_total_counts'].quantile(0.05)
    p95 = subset['log1p_total_counts'].quantile(0.95)
    print(f"\nAmostra {smp}:")
    print(f" - Sugestão de limite inferior (5%): {p5:.2f}")
    print(f" - Sugestão de limite superior (95%): {p95:.2f}")

Resultados:
* mediana: 5,87 PM e 4,79 PC. PM apresenta mais UMIS por célula que o controle
* limites propostos com intenção de equilibrar a limpeza do ruído sem perder tecido:
  * inferior: 3 (preserva ~95% PC e 99% PM)
  * superior: 8 (99% de ambas está abaixo disso)

In [ ]:
# visualizações pós corte
# visualização do total de UMIs com limites baseados nos percentis reais
plt.figure(figsize=(10, 5))
ax = sc.pl.violin(adata, keys=["log1p_total_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra")
ax.set_ylabel("Contagem de UMI (Log1p)")

# Limites baseados na análise numérica:
plt.axhline(y=3.0, color='r', linestyle='--', label="Limite Inferior (Corte Cauda PC)")
plt.axhline(y=8.0, color='r', linestyle='--', label="Limite Superior (Outliers)")

plt.title("Distribuição de UMI (Log1p) - Cortes Estatísticos")
plt.legend()
plt.show()

# visualização do percentual mitocondrial com limite de 20%
plt.figure(figsize=(10, 5))
ax = sc.pl.violin(adata, keys=["pct_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
plt.axhline(y=20.0, color='r', linestyle='--', label="Limite 20% (Morte Celular)")
plt.title("Percentual Mitocondrial - Corte Sugerido")
plt.legend()
plt.show()

In [ ]:
#aplica cortes
LOWER_LOG1P = 3.0
UPPER_LOG1P = 8.0
MAX_MT = 20.0

#filtra o objeto em memória
adata = adata[
    (adata.obs['log1p_total_counts'] >= LOWER_LOG1P) &
    (adata.obs['log1p_total_counts'] <= UPPER_LOG1P) &
    (adata.obs['pct_counts_mt'] <= MAX_MT)
].copy()

#df comparativo
path_raw = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_raw.zarr'
sdata_temp = spd.read_zarr(path_raw)
contagem_pre = sdata_temp.tables['segmentation_counts'].obs['sample'].value_counts()

#contagem do adata filtrado
contagem_pos = adata.obs['sample'].value_counts()

#monta df com amostras
df_qc = pd.DataFrame({
    'Amostra': contagem_pre.index,
    'Bins originais': contagem_pre.values,
    'Bins filtrados': [contagem_pos.get(s, 0) for s in contagem_pre.index]
})

#calcula impacto
df_qc['Retenção (%)'] = (df_qc['Bins filtrados'] / df_qc['Bins originais'] * 100).round(2)
df_qc['Perda (%)'] = (100 - df_qc['Retenção (%)']).round(2)

print("\nQC: impacto da filtragem (Estudo de Caso MYRF)")
display(df_qc.style.hide(axis='index').background_gradient(subset=['Perda (%)'], cmap='Reds'))

total_removido = df_qc['Bins originais'].sum() - df_qc['Bins filtrados'].sum()
print(f"\nTotal removido: {total_removido:,} bins.".replace(',', '.'))

#limpeza
del sdata_temp
gc.collect()

Resultados:
* a perda mínima de 2,43% do PM confirma a qualidade técnica altíssima da amostra, com sinal biológico muito superior ao ruído de fundo.
* perda de 11,48% do controle é esperada, visto que a cauda de baixa contagem era mais longa. A remoção desses bins garante que as comparações estatísticas futuras não sejam distorcidas por spots com pouca informação.

#**4. PCA**

In [ ]:
%%time

path_spm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm'

#carrega o checkpoint
checkpoint_zarr = os.path.join(path_spm, 'sdata_spm_raw.zarr')
concatenated_sdata = spd.read_zarr(checkpoint_zarr)

#extrai anndata
adata = concatenated_sdata.tables['segmentation_counts'].copy()
print(f"Total de observações carregadas: {adata.n_obs} bins.")

#QC - identifica mitocondrias
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, percent_top=None)

#QC - genes em menos de 50 bins
sc.pp.filter_genes(adata, min_cells=50)

#QC - cortes em log3 e log8
min_counts, max_counts = np.expm1(3.0).astype("int"), np.expm1(8.0).astype("int")
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)

#QC - filtro remove pontos em que o sinal mitocondrial é maior que 20%
adata = adata[adata.obs['pct_counts_mt'] <= 20.0].copy()

#normalização e seleção de HVG

# target_sum=1e4 para manter a consistência com a escala de profundidade do protocolo
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

#n_top_genes=10000 para capturar a complexidade necessária para identificar estágios de maturação de Sertoli e Leydig
sc.pp.highly_variable_genes(adata, n_top_genes=10000, flavor="seurat", batch_key="sample")

#redução de dimensionalidade - PCA
#svd_solver='arpack' é mais eficiente para o volume de dados
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)

#elbow plot
plt.figure(figsize=(12, 6))
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)
plt.show()

#checkpoint
save_path_h5ad = os.path.join(path_spm, 'adata_spm_pca_v1.h5ad')
adata.write_h5ad(save_path_h5ad)

#limpa RAM
gc.collect()

Resultados
* ainda que a diferença entre as células aqui seja apenas em relação ao nível de maturação e não de células normais vs células mutantes como no caso do benchmarking, optou-se por utilizar o HVG como um filtro frouxo. 10k genes ainda representa aproximadamente metade do transcriptoma detectável, evitando apenas genes onipresentes;
* embora o cotovelo seja gritante, o padrão da literatura de pacotes como Scanpy e Seurat é de 30 a 50 PCs. Como o que se busca é uma diferença muito fina (estágios de maturação de células), os 50 componentes foram mantidos.

#**5. Experimentos**

* após análise exploratória inicial, identificou-se um efeito de lote acentuado entre as amostras, o que evidenciou a necessidade da integração das amostras para que o clustering identifique populações celulares por sua assinatura transcriptômca e não pela origem da lâmina;
* nesta etapa, foram testados diferentes hiperparâmetros, aplicando diferentes intensidades de integração (valores de θ do Harmony) para encontrar o ponto de equilíbrio entre a correção técnica e a preservação do sinal biológico;
* partindo do `adata_spm_pca_v1.h5ad` gerado na seção anterior, foram testados inicialmente três cenários:
  * θ = 2: integração leve, priorizando a separação biológica original;
  * θ = 4: equilíbrio entre a mistura de amostras e a conservação de nichos;
  * θ = 8: intregação agressiva para casos de elevado ruído técnico;
* fixou-se a resolução do clustering em 2.5, granularidade necessária para capturar a complexidade da esteroidogênese. A pesquisadora forneceu uma lista com mais de 30 subpopulações de interesse, portanto, busca-se número igual ou maior de clusters nesta etapa;
* a escolha do modelo final será baseada em uma matriz de decisão composta por:
  * mistura de amostras, verificando a % de composição dos clusters, buscando clusters com misturas de 50/50 em populações estromais e vasculares;
  * identidade biológica, considerando a capacidade do modelo em separar tipos celulares distintos;
  * preservação do fenótipo patológico, mantendo clusters exclusivos para PM em linhagens em que a mutação comprovadamente altera o perfil celular, indicando que a integração não apagou os efeitos da condição;
* fixou-se sigma = 0,15 para manter a separação biológica em integrações muito fortes. É o fator que controla a suavidade da distribuição dos clusters no Harmony, definindo a largura da função gaussiana que calcula a probabilidade de uma célula pertencer a um determinado grupo. Quanto maior o sigma, mais flexível a integração e mais fácil de misturar, mas correndo risco de fundir tipos celulares diferentes. Quanto menor o sigma, mais seletiva a célula e só tem alta probabilidade de pertencer ao cluster que está fisicamente muito próximo dela no espaço latente;. Para θ = 8, é forçado que as amostras se sobreponham e se sigma fosse alto (o padrão é 0.1) o Harmony teria muita liberdade para mover as células entre clusters para forçar a sobreposição. Fixando sigma num valor mais baixo, cria-se uma resistência biológica, gerando fidelidade ao cluster obrigadno a célula a manter conexão forte com sua identidade origina e controlando supercorreção.

##**5.1. Integração e clustering**

In [ ]:
%%time

#caminhos e carga do PCA
path_spm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm'
input_path = os.path.join(path_spm, 'adata_spm_pca_v1.h5ad')
adata_base = sc.read_h5ad(input_path)

comparativo_composicao = {}

#intensidades de θ
intensidades = [2, 4, 8]

for theta in intensidades:
    print(f"\n--- Harmony θ = {theta} ---")

    #cópia para não sujar o objeto base no próximo loop
    adata = adata_base.copy()

    #integração
    #sigma = 0.15 para manter a separação biológica em integrações fortes
    sce.pp.harmony_integrate(adata, key='sample', basis='X_pca',
                             adjusted_basis='X_pca_harmony',
                             theta=theta, sigma=0.15)

    #vizinhos e UMAP
    # n_neighbors=50 para criar pontes suficientes entre PC e PM
    sc.pp.neighbors(adata, n_neighbors=50, use_rep="X_pca_harmony")
    sc.tl.umap(adata, min_dist=0.1, spread=1.0)

    #clustering - resolução 2.5
    sc.tl.leiden(adata, resolution=2.5, key_added="clusters",
                flavor="igraph", directed=False, random_state=0)

    #cálculo de composição para a matriz de decisão
    comp = adata.obs.groupby(['clusters', 'sample']).size().unstack(fill_value=0)
    comp_pct = (comp.div(comp.sum(axis=1), axis=0) * 100).round(2)
    comparativo_composicao[f"Theta_{theta}"] = comp_pct

    #checkpoint
    save_name = f'adata_spm_harmony_theta{theta}_res2.5.h5ad'
    adata.write_h5ad(os.path.join(path_spm, save_name))

    #visualização
    sc.pl.umap(adata, color=['sample', 'clusters'],
               title=[f'Efeito Lote (Theta {theta})', f'Clusters (Res 2.5 - Theta {theta})'])

    #limpa memória para o próximo loop
    del adata
    gc.collect()

Houve instabilidade no ambiente após a 2 loops, então o último foi rodado na célula a seguir

In [ ]:
%%time

#caminhos e carga do PCA
path_spm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm'
input_path = os.path.join(path_spm, 'adata_spm_pca_v1.h5ad')

#remove o copy()
adata = sc.read_h5ad(input_path)

theta = 8

print(f"\n--- Harmony θ = {theta} ---")

#integração
#sigma = 0.15 para manter a separação biológica em integrações fortes
sce.pp.harmony_integrate(adata, key='sample', basis='X_pca',
                         adjusted_basis='X_pca_harmony',
                         theta=theta, sigma=0.15)

#vizinhos e UMAP
# n_neighbors=50 para criar pontes suficientes entre PC e PM
sc.pp.neighbors(adata, n_neighbors=50, use_rep="X_pca_harmony")
sc.tl.umap(adata, min_dist=0.1, spread=1.0)

#clustering - resolução 2.5
sc.tl.leiden(adata, resolution=2.5, key_added="clusters",
            flavor="igraph", directed=False, random_state=0)

#cálculo de composição para a matriz de decisão
comp = adata.obs.groupby(['clusters', 'sample']).size().unstack(fill_value=0)
comp_pct = (comp.div(comp.sum(axis=1), axis=0) * 100).round(2)

print("Composição dos Clusters (%):")
print(comp_pct)

#checkpoint
save_name = f'adata_spm_harmony_theta{theta}_res2.5.h5ad'
adata.write_h5ad(os.path.join(path_spm, save_name))

#visualização
sc.pl.umap(adata, color=['sample', 'clusters'],
           title=[f'Efeito Lote (Theta {theta})', f'Clusters (Res 2.5 - Theta {theta})'])

#limpa memória
gc.collect()

Ainda que tenha 33 clusters identificados, fica claro pela composição que a maioria é quase 100% de um dos pacientes, o que indica que tem menos tipos celulares identificados de fato. Manteve-se theta = 8, que demonstrou melhor integração, e ajustou-se a resolução do Leiden para 4.0:


In [ ]:
#aumentando a resolução no objeto que está na memória
sc.tl.leiden(adata, resolution=4.0, key_added="clusters_res4",
            flavor="igraph", directed=False, random_state=0)

#composição - calculando contagem e porcentagem
comp_counts = adata.obs.groupby(['clusters_res4', 'sample']).size().unstack(fill_value=0)
comp_pcts = (comp_counts.div(comp_counts.sum(axis=1), axis=0) * 100).round(2)

#combinando em uma única tabela
comp_final = comp_counts.copy()
comp_final.columns = [f'{col}_n' for col in comp_final.columns]
comp_pcts.columns = [f'{col}_%' for col in comp_pcts.columns]
comp_final = pd.concat([comp_final, comp_pcts], axis=1)

print("Composição - resolução 4.0:")
display(comp_final)

#export
path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
comp_final.to_csv(os.path.join(path_tables, 'composicao_clusters_theta8_res4.0.csv'), index=True)

#visualização com figuras separadas para evitar sobreposição de legendas
#clusters
sc.pl.umap(adata, color='clusters_res4', title='Resolução 4.0 - Theta 8',
           legend_loc='right margin', frameon=True)

#amostra
sc.pl.umap(adata, color='sample', title='Amostra (Integrada)',
           legend_loc='right margin', frameon=True)

Resultados:
* clusters com n muito baixo podem ser tipos celulares muito raros ou apenas ruído de borda da lâmina. Clusters com mais bins tem muito mais relevância estatística e biológica;
* cluster 18 tem quase 10 mil bins no controle e praticamente zero na paciente. Se esse cluster for de espermatogônias ou células de Sertoli maduras, encontramos a diferença biológica causada pela mutação.


##5.2. Genes marcadores
* primeiro, identificaram-se os marcadores via Wilcoxon;
* depois, puxou-se os genes da planilha disponibilizada pela pesquisadora para confronta-los com os clusters.

Identificação de marcadores específicos via Wilcoxon:

In [ ]:
%%time

#genes marcadores -  pts=True calcula a fração de expressão (especificidade).
sc.tl.rank_genes_groups(adata=adata, groupby="clusters_res4", method="wilcoxon", pts=True)

#filtros de significância (p-val < 0.05)
df_markers_spm = sc.get.rank_genes_groups_df(adata=adata, group=None, pval_cutoff=0.05)

#filtro manual de magnitude
df_markers_spm = df_markers_spm[df_markers_spm['logfoldchanges'] >= 1.0]

#ordena por cluster e magnitude de expressão
df_markers_spm = df_markers_spm.sort_values(
    by=['group', 'logfoldchanges'],
    ascending=[True, False]
)

#checkpoint
export_path = os.path.join(path_spm, "marker_genes_spm_res4.0_v1.csv")
df_markers_spm.to_csv(export_path, index=False)

import gc
gc.collect()

Validação canônica: confronta os clusters com a lista de referência da pesquisadora, tratando as divergências de genes que não constam no dataset.
* standard_scale="var" - índice de confiança que normaliza a expressao do gene entre 0 e 1, quanto mais próximo de 1, maior a expressão do gene em comparação a todos os outros clusters

In [ ]:
%%time

#carrega planilha referência fornecida
path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')

#dicionário e filtro de segurança para genes presentes no dataset
marker_genes_raw = df_ref.groupby('Category')['Gene'].apply(list).to_dict()
marker_genes_pesquisadora = {}

for cat, genes in marker_genes_raw.items():
    valid_genes = [g for g in genes if g in adata.var_names]
    if valid_genes:
        marker_genes_pesquisadora[cat] = valid_genes

#dotplot
dp = sc.pl.dotplot(adata,
             var_names=marker_genes_pesquisadora,
             groupby="clusters_res4",
             standard_scale="var", #normaliza a expressão do gene entre 0 e 1
             title="Validação biológica com marcadores canônicos",
             return_fig=True) #extrai os dados em vez de plotar

#salva imagem
dp.savefig('_dotplot_validacao_canonica.png')

# 4. Exporta as matrizes de dados reais do gráfico
#intensidade da cor - Expressão média
dp.dot_color_df.to_csv(os.path.join(path_tables, "matriz_intensidade_canonica_res4.0.csv"))

#tamanho do ponto - células expressando o gene dentro do cluster
dp.dot_size_df.to_csv(os.path.join(path_tables, "matriz_tamanho_canonica_res4.0.csv"))

gc.collect()

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'

#carrega tabelas
df_comp = pd.read_csv(os.path.join(path_tables, 'composicao_clusters_theta8_res4.0.csv'), index_col=0)
df_int = pd.read_csv(os.path.join(path_tables, 'matriz_intensidade_canonica_res4.0.csv'), index_col=0)
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')

#dicionário
gene_to_category = dict(zip(df_ref['Gene'], df_ref['Category']))

#encontra gene dominante para cada cluster
df_int_numeric = df_int.select_dtypes(include='number')
top_genes = df_int_numeric.idxmax(axis=1)
top_scores = df_int_numeric.max(axis=1)

#gene vencedor da respectiva categoria
top_categories = top_genes.map(gene_to_category)

#formata indices dos clusters
df_comp.index = df_comp.index.astype(str)
top_genes.index = top_genes.index.astype(str)
top_categories.index = top_categories.index.astype(str)
top_scores.index = top_scores.index.astype(str)

#une informações na matriz final
df_final = df_comp.copy()
df_final['Categoria_Pesquisadora'] = top_categories
df_final['Gene_Marcador'] = top_genes
df_final['Score_Intensidade'] = top_scores.round(2)

#ordena pelo volume de células de PM
df_final = df_final.sort_values(by='PM_n', ascending=False)

print("Matriz de Decisão Final:")
display(df_final.head(15)) #top15

#export
output_file = os.path.join(path_tables, 'matriz_decisao_final_TCC.csv')
df_final.to_csv(output_file, index=True)

##**5.3. Marcadores nos clusters**

In [ ]:
%%time

path_spm = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm'
path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'

#carrega matriz de decisão
df_matriz = pd.read_csv(os.path.join(path_tables, 'matriz_decisao_final_TCC.csv'))

#garante que encontra a chave clusters_res4
if 'clusters_res4' in df_matriz.columns:
    chaves_clusters = df_matriz['clusters_res4'].astype(str)
else:
    chaves_clusters = df_matriz.index.astype(str)

#dicionario
mapeamento_spm = dict(zip(chaves_clusters, df_matriz['Categoria_Pesquisadora']))

#aplica mapeamento no adata
adata.obs["Categoria_Pesquisadora"] = adata.obs['clusters_res4'].astype(str).map(mapeamento_spm).astype('category')

#carrega o objeto SpatialData do projeto SPM
sdata_path = os.path.join(path_spm, 'sdata_spm_raw.zarr')
concatenated_sdata = spd.read_zarr(sdata_path)

#sincroniza as categorias com o objeto espacial
concatenated_sdata["segmentation_counts"].obs['Categoria_Pesquisadora'] = adata.obs['Categoria_Pesquisadora']

#salva sdata anotado
concatenated_sdata.write(os.path.join(path_spm, 'sdata_spm_anotado_v1.zarr'), overwrite=True)

#extensões da lamina
image_elements = list(concatenated_sdata.images.keys())
shape_elements = list(concatenated_sdata.shapes.keys())
extents = []

for i in range(len(image_elements)):
    extent = spd.get_extent(concatenated_sdata, elements=[shape_elements[i]], coordinate_system='downscale_to_hires')
    extents.append(extent)

#mapas espaciais
for i in range(len(image_elements)):
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    title = image_elements[i].replace("_hires_tissue_image","")

    #plota
    (
        crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i])
        .pl.render_images(image_elements[i], alpha=0.3)
        .pl.render_shapes(shape_elements[i], color="Categoria_Pesquisadora")
        .pl.show(ax=ax, title=f"{title} - Identificação de Células")
    )

    plt.tight_layout()
    plt.show()

Foi definido um código de cores para células maduras e imaturas

In [ ]:
%%time

#define paleta de cores

# IMATURAS - tons frios
cores_imaturas = {
    'primordial_germ_cells': '#1f77b4',
    'sertoli': '#2ca02c',
    'leydig': '#00ced1',
    'spermatogonia_undifferentiated': '#9467bd'
}

# MADURAS - tons quentes
cores_maduras = {
    'spermatozoa': '#d62728',
    'spermatid': '#e377c2',
    'sertoli_mature': '#ff7f0e'
}

#une selecoes e cria lista a ser exibida
palette_tcc = {**cores_imaturas, **cores_maduras}
categorias_selecionadas = list(palette_tcc.keys())

#gera mapas espaciais
for i in range(len(image_elements)):
    fig, ax = plt.subplots(1, 2, figsize=(20, 10)) # Atualizado para 2 colunas
    sample_name = image_elements[i].replace("_hires_tissue_image","") #identifica amostra

    # Histologia pura (Lado Esquerdo)
    (
        crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i])
        .pl.render_images(image_elements[i]) # Sem alpha, mostra a imagem H&E original
        .pl.show(ax=ax[0], title=f"{sample_name} - Histologia (H&E)")
    )

    # Mapa molecular com anotações (Lado Direito)
    (
        crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i])
        .pl.render_images(image_elements[i], alpha=0.2)
        .pl.render_shapes(
            shape_elements[i],
            color="Categoria_Pesquisadora",
            groups=categorias_selecionadas, #mostra apenas células listadas
            palette=palette_tcc,
            alpha=0.9
        )
        .pl.show(ax=ax[1], title=f"Mapa de Maturação: {sample_name}")
    )

    plt.tight_layout()
    plt.savefig(f'mapa_maturacao_comparativo_{sample_name}.png', dpi=300)
    plt.show()

##**5.4. Análise de expressão gênica diferencial**

Serão isolados tipos celulares específicos e comparar seu estado interno no PC contra PM.

Os resultadsos obtidos foram cruzados com os dados de single cell do [proteinatlas.org](https://www.proteinatlas.org/about/download#single_cell_type) para se obter os genes expressos. Os dados foram importados diretamente do site e preparados para as análises subsequentes.

In [ ]:
%%time
import urllib.request
import zipfile

#caminhos
url_hpa = "https://v23.proteinatlas.org/download/rna_single_cell_type.tsv.zip"
file_zip = "rna_single_cell_type.tsv.zip"
file_tsv = "rna_single_cell_type.tsv"

#download e extração se não estiver no colab
if not os.path.exists(file_tsv):
    urllib.request.urlretrieve(url_hpa, file_zip)
    with zipfile.ZipFile(file_zip, 'r') as zip_ref:
        zip_ref.extractall(".")

#carrega a base
df_hpa_full = pd.read_csv(file_tsv, sep='\t')

#gene | célula onde ele é mais expresso no corpo todo
idx_max = df_hpa_full.groupby('Gene name')['nTPM'].idxmax()
mapa_hpa_oficial = dict(zip(df_hpa_full.loc[idx_max]['Gene name'],
                            df_hpa_full.loc[idx_max]['Cell type']))

print(f"Concluído. {len(mapa_hpa_oficial)} genes mapeados com sucesso.")

In [ ]:
display(df_hpa_full.head(10))

###**5.4.1. Leydig**

PM apresentou milhares de células a mais em relação ao PC.
Os genes pouco expressos podem mostrar que faltam as enzimas que de fato fabricam a testosterona madura.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'leydig'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Consideração inicial: células de Leydig localizam-se no compartimento intersticial, externamente aos túbulos seminíferos, sendo as principais responsáveis pela regulação da função reprodutiva e a manutenção da espermatogênese

Resultados:
* 14.504 células para PM e apenas 410 para PC revela um desvio estrutural enorme na composição do tecido
* os top 10 genes perdidos pelo PM são Spermatocytes (estágio intermediário) e Late spermatids (estágio final), indicando o bloqueio da produção de espermatozoides, não há células em divisão. LFC < -1 revelam a redução extrema desses tipos celulares;
* o silenciamento dessas assinaturas de estágio intermediário e final da linhagem de Leydig indicam evidência quantitativa de que os túbulos adjacentes não possuem células germinativas em estágios avançados de maturação celular, corroborando com a hipótese de que o PM possui um perfil tecidual imaturo;
* a aplicação do teste de Wilcoxon permitiu validar quantitativamente o colapso do microambiente reprodutivo.


###**5.4.2. Sertoli**

As células de Sertoli são fundamentais para criar o ambiente em que o espermatozoide nasce, busca-se provar aqui que elas encontram-se num estado imaturo. Acredita-se que o MYRF atue fortemente aqui.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'sertoli'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Consideração inicial: as células de Sertoli situam-se no interior dos túbulos seminíferos e são essenciais para a formação testicular e a espermatogênese, atuando no suporte das células germinativas e no controle do ambiente tubular

Resultados:
*  PM: 18805; PC: 332 células, assim como para as células de Leydig, o PM tem um número absurdamente maior de células, indicando desvio estrutural;
* no top 10, APOA1 é um marcador de Sertoli fetal/imatura e 3 genes começam com MT-, indicando que são mitocondriais;
* os principais tipos celulares perdidos são Late spermatids, Spermatocytes e Spermatogonia, validando a ausência de células germinativas no PM;
* Considerando que as células de Sertoli atual como suporte físico direto das células germinativas, a ausência desse grupo fornece validação quantitativa da ausência de linha germinativa em estágios avançados de maturação

###5.4.3. Espermatogônias

Células germinativas localizadas nos túbulos seminíferos, são o ponto de partida da espermatogênese. Se multiplicam por mitose gerando espermatócitos que, por meiose, originam espermatozóides. Deseja-se observar se as espermatogônias de PM estão recebendo sinais de apoptose (morte celular programada) ou inflamação extrema, o que as impediria de evoluir.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'spermatogonia'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Resultados:
* PC: 39.293; PM: 5.668, quantidade drásticamente menor de espermatogônias no PM, diferente das células de Sertoli e Leydig;
* ganharam: basicamente células de Leydig. Mesmo num número muito menor nesse grupo, elas invadiram o espaço aqui;
* perdidos: Spermatocytes, Late spermatids e Early spermatids. No PC, houve uma progessão natural, enquanto que em PM esse sinal desapareceu. Mais uma evidência quantitativa.

###5.4.4. Espermatócitos

Células germinativas intermediárias na espermatogênese, localizadas nos túbulos seminíferos, que transformam-se em espermatozoides por meiose.

O objetivo deste bloco é identificar se PM consegue sustentar essa fasa da espermatogênese.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'spermatocyte'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Consideração: espermatócitos são o estágio intermediário

Resultados:
* PM: 11744 | PC: 6354,
* ganharam: vários genes mitocondriais, indicando estresse metabólico, COL1A2 de colágeno indicando fibrose
* perdidas: Spermatocytes, Late spermatids, Early spermatids. O modelo valida qu eo agrupamento perdeu as variáveis que fariam transição para a próxima fase da espermatogênese, o bloqueio é absoluto;
* espermatogônias (5668) > espermatócitos (6354) > espermátides > espermatozoides

###5.4.5. Espermatozoides

PM possui baixíssima quantidade de espermatozoides. O objetivo é avaliar a qualidade.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'spermatozoa'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Resultados:
* PC: 22579 | PM: 3592;
* ganhos: Leydig e estresse mitocondrial, mesmo perfil das fases anteriores;
* perdidos: Late spermatids, Spermatocytes

###5.4.6. Somáticas

O objetivo é demonstrar que a mutação no MYRF não só afeta uma célula específica, mas toda a estrutura do testículo.

In [ ]:
%%time

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/spm'
sdata_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/spm/sdata_spm_anotado_v1.zarr'

#busca adata
concatenated_sdata = spd.read_zarr(sdata_path)
adata = concatenated_sdata["segmentation_counts"]

#carrega a referência para mapear o grupo do gene
file_referencia = os.path.join(path_tables, 'goi_dsd_testis.xlsx')
df_ref = pd.read_excel(file_referencia, sheet_name='long_list')
mapa_categorias = dict(zip(df_ref['Gene'], df_ref['Category']))

#define palavra da linhagem
linhagem_alvo = 'somatic'

#pega qualquer categoria que tenha a palavra
celulas_selecionadas = adata.obs['Categoria_Pesquisadora'].astype(str).str.contains(linhagem_alvo, case=False, na=False)
adata_alvo = adata[celulas_selecionadas].copy()

#verifica se sobrou célula suficiente nos dois grupos
print(f"Células da linhagem '{linhagem_alvo}' encontradas:")
print(adata_alvo.obs['sample'].value_counts())

#normalização logarítmica
sc.pp.log1p(adata_alvo)

#Wilcoxon
sc.tl.rank_genes_groups(
    adata=adata_alvo,
    groupby='sample',
    reference='PC',   #controle como base
    method='wilcoxon',
    key_added='dge_macro'
)

#extrai os resultados brutos
df_dge = sc.get.rank_genes_groups_df(adata_alvo, group='PM', key='dge_macro')

#mapeamento de categorias e HPA
df_dge['Categoria_Gene'] = df_dge['names'].map(mapa_categorias).fillna('Outros/Nao_Listado')
df_dge['Identidade_HPA_Oficial'] = df_dge['names'].map(mapa_hpa_oficial).fillna('Nao_Detectado/Baixa_Expressao')

#extrai genes superexpressos em PM
df_dge_sig_pm = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] > 1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} ganhou em PM:")

#exibe coluna de validação
display(df_dge_sig_pm[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#extrai genes pouco expressos em PM (perdidos)
df_dge_sig_pc = df_dge[(df_dge['pvals_adj'] < 0.05) & (df_dge['logfoldchanges'] < -1.0)]

print(f"\nTop 10 Genes que a linhagem {linhagem_alvo} perdeu em PM:")
display(df_dge_sig_pc.sort_values('logfoldchanges')[['names', 'logfoldchanges', 'Categoria_Gene', 'Identidade_HPA_Oficial']].head(10))

#export
df_dge_sig_pm.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Superexpressos_PM_Validado.csv'), index=False)
df_dge_sig_pc.to_csv(os.path.join(path_tables, f'DGE_{linhagem_alvo}_Perdidos_PM_Validado.csv'), index=False)

Resultados:
* PM: 7730 | PC: 223: observou-se um aumento volumétrico acentuado no grupamento de células somáticas na amostra PM (7.730 bins) em relação à PC (223 bins). A análise de expressão diferencial revelou que este aumento é impulsionado pela superexpressão do gene DLK1 (LFC: 28,5), um marcador canônico de imaturidade em células de Leydig. Tais dados indicam que o cluster somático representa, na verdade, uma expansão do microambiente de Leydig imaturo, caracterizando um quadro de hiperplasia patológica.;
* observou-se a ausência de assinaturas transcricionais vinculadas às fases tardias da espermatogênese (Late spermatids), corroborando a hipótese de interrupção da maturação germinativa no PM.